# CrisisQ — practical impact, rule attribution, and matched CPU/device benchmarks

**This is the final targeted validation, not another training/search experiment.** It uses your latest frozen models and your completed external-public evaluation. Attach BOTH **full notebook outputs** as Kaggle Inputs. Everything is discovered recursively under `/kaggle/input`; no ZIP input or manual paths are needed.

The notebook answers two practical questions:
1. Which frozen rule categories add detections, introduce negative alerts, or remove alerts relative to the same neural model? This is a **post-hoc descriptive analysis** of already inspected data, not new confirmatory validation.
2. For the preselected primary model, what do FP32/INT8 and neural/hybrid choices actually cost on the SAME sampled messages? CPU timing measures routing, tokenization, tensor preparation, inference and thresholding. It also measures session initialization and isolated-process peak RSS separately.

**Kaggle CPU timing is not phone timing.** A small optional handoff packet includes the same cases, token IDs, expected outputs, a verified Dart gate, a Dart timing adapter for your existing Flutter app, and a CSV import contract. You must execute it on the phone before claiming phone speed, RAM or battery benefits. The adapter intentionally takes your existing local tokenizer and ONNX calls as callbacks; it is not a standalone Flutter app and does not choose a new runtime for you.

No rules, weights, quantizers, labels, thresholds or acceptance budgets are changed. No clinician annotation is required. Higher recall plus more alerts is reported as a tradeoff, not automatic benefit. Public source labels and the prior incomplete historical-overlap qualification remain unchanged.


In [ ]:
from pathlib import Path
INPUT_BASE=Path('/kaggle/input')
OUTPUT=Path('/kaggle/working/crisisq_practical_impact_v1')
PRIVATE=Path('/kaggle/temp/crisisq_practical_impact_v1')
EXPECTED_EXTERNAL_HASH='fe7125e21acb826cd46fa837388b55ff75dfe1b4177a68cdebf133d14ab1dfbf'
EXPECTED_SELECTION_HASH='e976dbf922c37ce1406418ff9c5903a1ecc6a502aadb14d3d116ab18ef6ba626'
RUN_CPU_BENCHMARK=True
MESSAGES_PER_PROFILE=64
REPEATS=5
WARMUP_MESSAGES=10
ORDER_SEED=20261111
# The small phone adapter packet is exported automatically; weights are excluded by default.
INCLUDE_PHONE_WEIGHTS=False  # optional packaging only; ~335 MB plus any ONNX sidecars
DEVICE_RESULTS_CSV=None     # optional path to logs from a real phone; importer checks the contract
for p in [OUTPUT/'tables',OUTPUT/'manifests',OUTPUT/'cpu_runs',OUTPUT/'phone_adapter',PRIVATE]:p.mkdir(parents=True,exist_ok=True)


## 1. Locate both source outputs and match inference versions
The expected external evaluation and original selection hashes are fixed to your verified runs. If either full output is missing, execution stops instead of substituting a model or retraining.


In [ ]:
import json,hashlib,sys,os,subprocess,importlib.metadata,platform

def _hash(x):return hashlib.sha256(json.dumps(x,sort_keys=True,default=str).encode()).hexdigest()
external=[];runs=[]
for p in sorted(INPUT_BASE.rglob('evaluation_frozen.json')):
    if p.parent.name!='manifests':continue
    try:
        obj=json.loads(p.read_text());root=p.parent.parent
        if _hash(obj)==EXPECTED_EXTERNAL_HASH and (root/'tables/predictions_without_text.csv').exists():external.append((root,obj))
    except (ValueError,OSError):pass
for p in sorted(INPUT_BASE.rglob('selection_frozen.json')):
    q=p.parent/'protocol_frozen.json'
    if not q.exists():continue
    try:
        selection=json.loads(p.read_text());protocol=json.loads(q.read_text())
        if _hash(selection)==EXPECTED_SELECTION_HASH and selection.get('protocol')==_hash(protocol) and (p.parent/'models').is_dir():runs.append((p.parent,selection,protocol))
    except (ValueError,OSError):pass
if not external or not runs:
    raise FileNotFoundError('Attach BOTH full notebook outputs: (1) frozen nonclinical models, and (2) completed external-public evaluation. Reports-only ZIP files do not contain model weights.')
EXTERNAL,EXTERNAL_PLAN=external[0];RUN,SELECTION,PROTOCOL=runs[0]
if EXTERNAL_PLAN['source_selection_hash']!=EXPECTED_SELECTION_HASH:raise RuntimeError('External evaluation belongs to a different model selection')
print('Model input:',RUN);print('External results:',EXTERNAL)
required={'numpy':None,'pandas':None,'scipy':None,'scikit-learn':None,
 'onnxruntime':PROTOCOL['versions']['onnxruntime'],'onnx':PROTOCOL['versions']['onnx'],
 'transformers':PROTOCOL['versions']['transformers']}
install=[]
for pkg,wanted in required.items():
    try:current=importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:current=None
    if current is None or (wanted is not None and current!=wanted):
        module='sklearn' if pkg=='scikit-learn' else pkg
        if module in sys.modules:raise RuntimeError(f'Restart the kernel before changing already imported {pkg}')
        install.append(pkg+('=='+wanted if wanted else ''))
if install:subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-cache-dir']+install)


In [ ]:
"""Audited inference-only utilities embedded in the notebook."""
from pathlib import Path
import json, hashlib, re, unicodedata, html, math, os, urllib.request, time, zipfile
import numpy as np
import pandas as pd
from scipy.stats import beta, t as student_t
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score, average_precision_score


def digest(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,default=str).encode()).hexdigest()

def file_sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()

def atomic_json(path,obj):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,sort_keys=True,default=str)); tmp.replace(path)

def freeze(path,obj):
    path=Path(path)
    if path.exists():
        if json.loads(path.read_text())!=json.loads(json.dumps(obj,default=str)):
            raise RuntimeError(f'{path.name} differs from the frozen run. Preserve the existing run; do not overwrite or retune after results.')
    else: atomic_json(path,obj)
    return digest(obj)

def normal_text(text):
    s=unicodedata.normalize('NFKC',html.unescape(str(text))).casefold()
    s=s.replace('\u2019',"'")
    return re.sub(r'\s+',' ',re.sub(r'[^\w\s]',' ',s)).strip()

def text_hash(text): return hashlib.sha256(normal_text(text).encode()).hexdigest()

def source_frame_digest(frame):
    # Reproduce the previous notebook's read_frame + frame_digest convention exactly.
    d=frame.copy()
    d['text']=d.text.astype(str); d['label']=pd.to_numeric(d.label,errors='raise').astype(int)
    oldhash=d.text.map(lambda s:hashlib.sha256(re.sub(r'\s+',' ',str(s)).strip().lower().encode()).hexdigest())
    if 'group_id' not in d: d['group_id']=d['group'].astype(str) if 'group' in d else oldhash
    for c,v in [('profile','real_world'),('source','unspecified')]:
        if c not in d: d[c]=v
    return digest(d[['text','label','group_id','profile','source']].to_dict('records'))

def discover_run(base,expected_hash):
    candidates=[]; seen=[]
    for selection in sorted(Path(base).rglob('selection_frozen.json')):
        root=selection.parent; protocol=root/'protocol_frozen.json'
        if not protocol.exists(): continue
        try:
            s=json.loads(selection.read_text()); p=json.loads(protocol.read_text())
            sh=digest(s); ph=digest(p)
            if s.get('protocol')!=ph or sh!=expected_hash: continue
            seen.append(str(root))
            if (root/'models').is_dir(): candidates.append((root,s,p,sh,ph))
        except (ValueError,OSError): continue
    if not candidates:
        raise FileNotFoundError('No matching frozen run with models/ found. Attach the FULL latest notebook output as Kaggle Input; the reports ZIP alone has no weights. Matching manifest roots seen: '+str(seen))
    # Duplicate mounts with identical frozen selection are safe; never select by evaluation performance.
    return candidates[0]


def build_gate(spec):
    # Rebuild from frozen data; never execute source code read from an attachment.
    required={'crisis','safe','negators','contrast','risk','v1_crisis','v1_safe'}
    if not required<=set(spec): raise ValueError('Frozen rule specification is incomplete')
    C=[(i,re.compile(p)) for i,c,p in spec['crisis']]
    S=[(i,re.compile(p)) for i,c,p in spec['safe']]
    negators=set(spec['negators']); contrast=re.compile(spec['contrast']); risk=re.compile(spec['risk'])
    def normalize(text):
        s=str(text).lower()
        for ch in ["'",'’','`','‘']: s=s.replace(ch,'')
        return re.sub(r'\s+',' ',re.sub(r'[^a-z0-9\s]',' ',s)).strip()
    def route(text,version='v2',trace=False):
        s=normalize(text)
        if version=='v1':
            for p in spec['v1_safe']:
                if normalize(p) in s: return (0,'legacy_safe') if trace else 0
            for p in spec['v1_crisis']:
                if normalize(p) in s: return (1,'legacy_crisis') if trace else 1
            return (-1,'uncertain') if trace else -1
        aff=[]; neg=[]; safe=[]; spans=[]
        for rid,rx in C:
            for m in rx.finditer(s):
                (neg if any(w in negators for w in s[:m.start()].split()[-4:]) else aff).append(rid)
        for rid,rx in S:
            for m in rx.finditer(s): safe.append(rid); spans.append(m.span())
        if aff and safe: out,reason=-1,'conflict_to_stage2'
        elif aff: out,reason=1,'crisis'
        elif neg: out,reason=-1,'negated_crisis_to_stage2'
        elif safe:
            residual=s
            for a,b in sorted(spans,reverse=True): residual=residual[:a]+' '+residual[b:]
            if contrast.search(s) or risk.search(residual): out,reason=-1,'safe_with_residual_risk_to_stage2'
            else: out,reason=0,'safe_context'
        else: out,reason=-1,'uncertain'
        return (out,reason) if trace else out
    return route


def download_pinned(repo,commit,name,directory):
    from urllib.parse import quote
    dest=Path(directory)/(repo.replace('/','__')+'__'+Path(name).name)
    dest.parent.mkdir(parents=True,exist_ok=True)
    url=f'https://raw.githubusercontent.com/{repo}/{commit}/'+quote(name)
    if not dest.exists():
        tmp=dest.with_suffix(dest.suffix+'.download')
        for attempt in range(3):
            try:
                req=urllib.request.Request(url,headers={'User-Agent':'CrisisQ-external-evaluation/1.0'})
                with urllib.request.urlopen(req,timeout=90) as r, tmp.open('wb') as f:
                    while True:
                        b=r.read(1024*1024)
                        if not b: break
                        f.write(b)
                tmp.replace(dest); break
            except Exception:
                if attempt==2: raise
                time.sleep(2)
    return dest,dict(repo=repo,commit=commit,path=name,url=url,sha256=file_sha(dest),bytes=dest.stat().st_size)


def parse_twitter(path):
    d=pd.read_csv(path,keep_default_na=False)
    if not {'tweet','intention'}<=set(d): raise ValueError('Public Twitter schema changed: expected tweet,intention')
    labels=pd.to_numeric(d.intention,errors='raise')
    if not labels.isin([0,1]).all(): raise ValueError('Unexpected Twitter labels; do not guess a mapping')
    return pd.DataFrame(dict(text=d.tweet.astype(str),label=labels.astype(int),source='twitter_intention',
        source_id=['twitter:'+str(i) for i in range(len(d))],group_id=['twitter:'+str(i) for i in range(len(d))],role='author'))


def parse_esconv(path):
    rows=[]; data=json.loads(Path(path).read_text())
    if not isinstance(data,list): raise ValueError('ESConv must be a list')
    for i,item in enumerate(data):
        dialog=item.get('dialog')
        if not isinstance(dialog,list): raise ValueError('ESConv dialog schema mismatch')
        for role in ['seeker','supporter']:
            # One first substantive utterance per role per conversation; not model-based selection.
            chosen=None
            for j,turn in enumerate(dialog):
                if turn.get('speaker')!=role: continue
                text=turn.get('content')
                if not isinstance(text,str): raise ValueError('Expected ESConv content text')
                if len(text.strip())>=30 and len(text.split())>=5:
                    chosen=(j,text); break
            if chosen:
                j,text=chosen
                rows.append(dict(text=text,label=-1,source='esconv',source_id=f'esconv:{i}:{j}',group_id=f'esconv:{i}',role=role))
    if not rows: raise ValueError('No ESConv turns parsed')
    return pd.DataFrame(rows)


def collect_old_texts(root,protocol):
    root=Path(root); frames=[]; inventory=[]; verified={}
    for p in sorted(root.rglob('*.csv')):
        if p.stat().st_size>150*1024**2: continue
        try:
            d=pd.read_csv(p,keep_default_na=False)
            col=next((c for c in ['text','tweet','content'] if c in d.columns),None)
            if col:
                vals=d[col].astype(str); vals=vals[vals.str.len()>=10]
                frames.append(pd.DataFrame(dict(text=vals,origin=str(p.relative_to(root)))))
                inventory.append(dict(file=str(p.relative_to(root)),rows=len(vals)))
            if col=='text' and 'label' in d:
                h=source_frame_digest(d)
                for split,wanted in protocol.get('splits',{}).items():
                    if h==wanted: verified[split]=str(p.relative_to(root))
        except (ValueError,TypeError,KeyError,pd.errors.ParserError): continue
    old=pd.concat(frames,ignore_index=True) if frames else pd.DataFrame(columns=['text','origin'])
    if len(old): old=old.drop_duplicates('text').reset_index(drop=True)
    expected=set(protocol.get('splits',{})); missing=sorted(expected-set(verified))
    return old,dict(verified_complete_splits=verified,missing_complete_splits=missing,
        complete_historical_archive=bool(expected) and not missing,known_unique_texts=len(old),files=inventory,
        conclusion='complete exact-text reference coverage; cross-post/author and pretraining overlap still not proven' if not missing else 'partial historical overlap audit: absence of overlap does not prove no training/evaluation contamination')


def deduplicate_and_audit(candidates,old,near_threshold=.90):
    d=candidates.copy(); d['text']=d.text.astype(str)
    d=d[d.text.str.strip().str.len().ge(20)&d.text.str.split().str.len().ge(3)].copy()
    d['text_hash']=d.text.map(text_hash)
    conflicts=d[d.label>=0].groupby('text_hash').label.nunique()
    conflicting=set(conflicts[conflicts>1].index)
    d=d[~d.text_hash.isin(conflicting)].copy()
    # Keep one deterministic representative per text; preserve all its original conversation links.
    d=d.sort_values('source_id').reset_index(drop=True)
    all_group_links=d.groupby('text_hash').group_id.apply(list).to_dict()
    d=d.drop_duplicates('text_hash').reset_index(drop=True)
    n=len(d); parent=list(range(n))
    def find(a):
        while parent[a]!=a:
            parent[a]=parent[parent[a]];a=parent[a]
        return a
    def union(a,b):
        a,b=find(a),find(b)
        if a!=b: parent[max(a,b)]=min(a,b)
    first={}
    for i,row in d.iterrows():
        for g in all_group_links[row.text_hash]:
            if g in first: union(i,first[g])
            else: first[g]=i
    oldtexts=old.text.astype(str).tolist() if len(old) else []
    oldhash=set(map(text_hash,oldtexts))
    exact=d.text_hash.isin(oldhash).to_numpy(); old_match={}; near_edges=0
    texts=oldtexts+d.text.tolist()
    if not texts: raise ValueError('No public evaluation texts')
    vec=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),max_features=100000,dtype=np.float32)
    z=vec.fit_transform(texts); oldz=z[:len(oldtexts)]; newz=z[len(oldtexts):]
    for start in range(0,n,96):
        block=newz[start:start+96]
        if len(oldtexts):
            sim=(block@oldz.T).tocsr()
            for local in range(block.shape[0]):
                row=sim.getrow(local)
                if row.nnz and row.data.max()>=near_threshold:
                    j=int(row.indices[row.data.argmax()]);old_match[start+local]=(j,float(row.data.max()))
        sim=(block@newz.T).tocoo()
        for a,b,v in zip(sim.row,sim.col,sim.data):
            i=start+int(a); j=int(b)
            if j>i and v>=near_threshold: union(i,j);near_edges+=1
    roots=[find(i) for i in range(n)]
    members={}
    for i,r in enumerate(roots): members.setdefault(r,[]).append(i)
    groupnames={r:'cluster:'+min(d.text_hash.iloc[i] for i in ii) for r,ii in members.items()}
    d['group_id']=[groupnames[r] for r in roots]
    contaminated={roots[i] for i in range(n) if exact[i] or i in old_match}
    flags=[]
    for i in range(n):
        if exact[i] or i in old_match:
            flags.append(dict(source_id=d.source_id.iloc[i],text_hash=d.text_hash.iloc[i],exact_known_overlap=bool(exact[i]),
                lexical_similarity=old_match.get(i,(None,None))[1],cluster=d.group_id.iloc[i]))
    removed=d[d.group_id.isin({groupnames[r] for r in contaminated})]
    clean=d.drop(removed.index).reset_index(drop=True)
    return clean,pd.DataFrame(flags,columns=['source_id','text_hash','exact_known_overlap','lexical_similarity','cluster']),dict(
        input_rows=len(candidates),label_conflict_hashes=len(conflicting),unique_eligible_before_overlap=n,
        exact_known_overlap_rows=int(exact.sum()),lexical_known_overlap_rows=len(old_match),
        candidate_near_duplicate_edges=near_edges,removed_rows_in_overlapping_groups=len(removed),remaining_rows=len(clean))


def draw_groups(pool,budgets,salt):
    # Whole duplicate/conversation groups go to one evaluation partition.
    rank=lambda g:hashlib.sha256((salt+'|'+str(g)).encode()).hexdigest()
    target={int(k):int(v) for k,v in budgets.items()}; counts={k:0 for k in target}; picked=[]
    for group in sorted(pool.group_id.unique(),key=rank):
        g=pool[pool.group_id==group]
        if not any(counts.get(int(y),0)<target.get(int(y),0) for y in g.label.unique()): continue
        picked.append(g)
        for y,c in g.label.value_counts().items(): counts[int(y)]=counts.get(int(y),0)+int(c)
        if all(counts.get(y,0)>=n for y,n in target.items()): break
    return (pd.concat(picked,ignore_index=True) if picked else pool.iloc[:0].copy()),counts


def metric(y,pred):
    y=np.asarray(y); pred=np.asarray(pred); valid=np.isin(y,[0,1]); pos=y==1; neg=y==0
    npos=int(pos.sum()); nneg=int(neg.sum()); tp=int(((pred==1)&pos).sum()); fp=int(((pred==1)&neg).sum())
    return dict(n=len(y),n_labeled=int(valid.sum()),n_positive=npos,n_negative=nneg,tp=tp,fn=npos-tp,fp=fp,tn=nneg-fp,
        recall=tp/npos if npos else None,fpr=fp/nneg if nneg else None,
        alert_rate=float(np.mean(pred==1)) if len(pred) else None,
        precision=tp/(tp+fp) if valid.any() and tp+fp else None)

def decide(p,tau,gate=None):
    out=(np.asarray(p)>=float(tau)).astype(int)
    return out if gate is None else np.where(np.asarray(gate)<0,out,gate).astype(int)

def exact_bounds(k,n):
    if not n: return None,None,None
    lo=0. if not k else float(beta.ppf(.025,k,n-k+1))
    hi=1. if k==n else float(beta.ppf(.975,k+1,n-k))
    upper=1. if k==n else float(beta.ppf(.95,k+1,n-k))
    return lo,hi,upper

def group_interval(values,groups,B=2000,seed=20261111):
    values=np.asarray(values,float); groups=np.asarray(groups)
    if not len(values): return None,None
    names=np.unique(groups)
    if len(names)<2: return None,None
    counts=np.array([(groups==g).sum() for g in names],float)
    sums=np.array([values[groups==g].sum() for g in names],float)
    rng=np.random.default_rng(seed); stats=[]
    for start in range(0,B,128):
        idx=rng.integers(0,len(names),size=(min(128,B-start),len(names)))
        stats.extend((sums[idx].sum(axis=1)/counts[idx].sum(axis=1)).tolist())
    return tuple(map(float,np.quantile(stats,[.025,.975])))

def paired_seed_interval(values):
    v=np.asarray(values,float);v=v[np.isfinite(v)]
    if not len(v): return None,None,None
    mean=float(v.mean())
    if len(v)<2: return mean,None,None
    half=float(student_t.ppf(.975,len(v)-1)*v.std(ddof=1)/np.sqrt(len(v)))
    return mean,mean-half,mean+half


def model_files(root,arm,seed,variant):
    directory=Path(root)/'models'/f'{arm}__{seed}'; model=directory/f'{variant}.onnx'
    if not model.exists(): raise FileNotFoundError(f'Missing frozen weight file {model}; no retraining or model substitution is allowed.')
    if not (directory/'tokenizer_config.json').exists() or not ((directory/'tokenizer.json').exists() or (directory/'vocab.txt').exists()):
        raise FileNotFoundError(f'Local tokenizer is missing in {directory}')
    import onnx
    graph=onnx.load(str(model),load_external_data=False)
    paths=[model]
    for tensor in graph.graph.initializer:
        if tensor.data_location==onnx.TensorProto.EXTERNAL:
            loc=next((e.value for e in tensor.external_data if e.key=='location'),None)
            if not loc: raise ValueError('External ONNX tensor has no location')
            p=(directory/loc).resolve()
            if directory.resolve() not in p.parents: raise ValueError('Unsafe external weight path')
            if not p.exists(): raise FileNotFoundError(f'Missing external weights: {p}')
            paths.append(p)
    paths.extend(p for p in directory.iterdir() if p.name in ['tokenizer_config.json','tokenizer.json','vocab.txt','special_tokens_map.json','config.json','added_tokens.json'])
    return directory,sorted(set(paths),key=str)


def infer_checkpointed(model_path,tokenizer_dir,encoded,path,signature,max_length,threads=4):
    import onnxruntime as ort
    path=Path(path); meta=path.with_suffix('.json'); shards=path.parent/(path.stem+'_shards')
    cache_meta=dict(signature=signature,n=len(encoded['input_ids']),batch=1,max_length=max_length)
    if path.exists():
        if not meta.exists() or json.loads(meta.read_text())!=cache_meta: raise RuntimeError('Prediction cache identity mismatch')
        p=np.load(path)
        if p.shape!=(cache_meta['n'],) or not np.isfinite(p).all(): raise RuntimeError('Corrupt probability cache')
        return p
    shards.mkdir(parents=True,exist_ok=True); freeze(shards/'identity.json',cache_meta)
    so=ort.SessionOptions(); so.intra_op_num_threads=threads
    session=ort.InferenceSession(str(model_path),sess_options=so,providers=['CPUExecutionProvider'])
    ins=session.get_inputs();names={i.name for i in ins}
    if not {'input_ids','attention_mask'}<=names or names-{'input_ids','attention_mask','token_type_ids'}:
        raise ValueError(f'Unexpected ONNX inputs: {names}')
    for i in ins:
        if i.type!='tensor(int64)': raise ValueError(f'Expected int64 input, got {i.type}')
        if isinstance(i.shape[1],int) and i.shape[1]!=max_length: raise ValueError('Frozen ONNX sequence length differs from protocol')
    outputs=session.get_outputs()
    if 'logits' not in {o.name for o in outputs}: raise ValueError('Expected named logits output')
    allp=[]; n=cache_meta['n']
    for start in range(0,n,128):
        end=min(start+128,n); chunk=shards/f'{start:08d}.npy'
        if chunk.exists():
            p=np.load(chunk)
            if p.shape!=(end-start,) or not np.isfinite(p).all(): raise RuntimeError(f'Corrupt shard {chunk}')
        else:
            vals=[]
            for i in range(start,end):
                feed={k:encoded[k][i:i+1].astype(np.int64) for k in names if k in encoded}
                if 'token_type_ids' in names and 'token_type_ids' not in feed: feed['token_type_ids']=np.zeros_like(feed['input_ids'])
                logits=np.asarray(session.run(['logits'],feed)[0],dtype=np.float64)
                if logits.shape!=(1,2): raise ValueError(f'Expected two logits with positive class index 1; got {logits.shape}')
                logits-=logits.max(axis=1,keepdims=True); e=np.exp(logits); vals.append(float((e/e.sum(axis=1,keepdims=True))[0,1]))
            p=np.asarray(vals); tmp=chunk.with_suffix('.tmp.npy');np.save(tmp,p);tmp.replace(chunk)
        allp.extend(p.tolist())
        if start%512==0: print(f'  inference {end}/{n}',flush=True)
    p=np.asarray(allp)
    if not np.isfinite(p).all() or ((p<0)|(p>1)).any(): raise RuntimeError('Invalid probabilities')
    tmp=path.with_suffix('.tmp.npy');np.save(tmp,p);tmp.replace(path);atomic_json(meta,cache_meta)
    del session
    return p


In [ ]:
import gc,shutil,time,csv
import onnxruntime as ort
from transformers import AutoTokenizer
for pkg in ['onnxruntime','onnx','transformers']:
    if importlib.metadata.version(pkg)!=PROTOCOL['versions'][pkg]:raise RuntimeError('Inference package mismatch; restart kernel')
MAX_LENGTH=int(PROTOCOL['max_length']);THREADS=min(4,os.cpu_count() or 1)
PRIMARY=EXTERNAL_PLAN['primary']
if PRIMARY is None:raise RuntimeError('No predeclared accepted SAQS configuration exists')
ARM=PRIMARY['arm'];SEED=int(PRIMARY['seed']);VARIANT=PRIMARY['variant'];TAU=float(PRIMARY['tau'])
if SELECTION['models'][f'{ARM}__{SEED}']['selected']!={'variant':VARIANT,'tau':TAU}:raise RuntimeError('Primary configuration changed')
INDEX=pd.read_csv(EXTERNAL/'tables/evaluation_row_index.csv')
PRED=pd.read_csv(EXTERNAL/'tables/predictions_without_text.csv')
if digest(INDEX[['source_id','text_hash','label','group_id','partition','role']].to_dict('records'))!=EXTERNAL_PLAN['evaluation_rows']:
    raise RuntimeError('Frozen external row index mismatch')
GATE=build_gate(PROTOCOL['rules'])
print('Preselected primary:',PRIMARY)
print('No new classifier-performance experiment will be used to select a replacement configuration.')


## 2. Reconstruct only the already evaluated public texts
Public files are downloaded at the SAME Git commits and byte hashes saved by the external evaluation. Original row IDs reconstruct precisely the existing evaluated rows; no resampling or new exclusion is allowed. Source text is kept under `/kaggle/temp`. The ordinary results ZIP contains identifiers and counts, not raw posts.


In [ ]:
source_paths={}
for name,record in EXTERNAL_PLAN['source_records'].items():
    p,meta=download_pinned(record['repo'],record['commit'],record['path'],PRIVATE)
    if meta['sha256']!=record['sha256']:raise RuntimeError(f'Public source changed: {name}')
    source_paths[name]=p
raw_twitter=pd.read_csv(source_paths['twitter'],keep_default_na=False)
raw_es=json.loads(Path(source_paths['esconv']).read_text())
def recover_text(source_id):
    parts=source_id.split(':')
    if parts[0]=='twitter':return str(raw_twitter.iloc[int(parts[1])]['tweet'])
    if parts[0]=='esconv':return str(raw_es[int(parts[1])]['dialog'][int(parts[2])]['content'])
    raise ValueError('Unknown frozen source ID')
E=INDEX.copy();E['text']=[recover_text(s) for s in E.source_id]
if not (E.text.map(text_hash)==E.text_hash).all():raise RuntimeError('Recovered texts do not match frozen external hashes')
if not np.array_equal(np.array([GATE(t) for t in E.text]),E.gate_v2.to_numpy()):raise RuntimeError('Frozen routing reconstruction differs')
print('Recovered and verified existing external rows:',len(E))


## 3. Rule traces and exact routing transitions (existing caches only)
Per-rule hit tables are associations: several rules can match the same text, so hit counts must not be summed as disjoint causal effects. The unique-rule column isolates cases with exactly one affirmed crisis rule. Pipeline transitions are exact comparisons with the same weights and threshold. All false-positive terminology is relative to the existing source labels, not clinical truth. ESConv has no crisis labels and remains an unlabeled alert analysis.


In [ ]:
def rule_trace(text):
    t=str(text).lower()
    for ch in ["'",'’','`','‘']:t=t.replace(ch,'')
    t=re.sub(r'\s+',' ',re.sub(r'[^a-z0-9\s]',' ',t)).strip()
    affirmed=[];negated=[];safe=[];spec=PROTOCOL['rules']
    for rid,category,pattern in spec['crisis']:
        for m in re.finditer(pattern,t):
            (negated if any(w in set(spec['negators']) for w in t[:m.start()].split()[-4:]) else affirmed).append(rid)
    for rid,category,pattern in spec['safe']:
        if re.search(pattern,t):safe.append(rid)
    return dict(affirmed=sorted(set(affirmed)),negated=sorted(set(negated)),safe=sorted(set(safe)))
TRACES={row.source_id:rule_trace(row.text) for row in E.itertuples()}
trace_rows=[dict(source_id=s,**{k:'|'.join(v) for k,v in t.items()}) for s,t in TRACES.items()]
pd.DataFrame(trace_rows).to_csv(OUTPUT/'tables/frozen_rule_traces_without_text.csv',index=False)
NEURAL=PRED[(PRED.analysis=='source_selected')&(PRED.system=='neural')]
HYBRID=PRED[(PRED.analysis=='source_selected')&(PRED.system=='hybrid_v2')]
keys=['job','arm','seed','variant','analysis','tau','source_id','partition','label']
PAIRS=NEURAL[keys+['prediction']].merge(HYBRID[keys+['prediction']],on=keys,suffixes=('_neural','_hybrid'),validate='one_to_one')
if len(PAIRS)!=len(NEURAL):raise RuntimeError('Unpaired predictions in source results')
PAIRS['positive_gained']=(PAIRS.label==1)&(PAIRS.prediction_neural==0)&(PAIRS.prediction_hybrid==1)
PAIRS['positive_lost']=(PAIRS.label==1)&(PAIRS.prediction_neural==1)&(PAIRS.prediction_hybrid==0)
PAIRS['negative_alert_added']=(PAIRS.label==0)&(PAIRS.prediction_neural==0)&(PAIRS.prediction_hybrid==1)
PAIRS['negative_alert_removed']=(PAIRS.label==0)&(PAIRS.prediction_neural==1)&(PAIRS.prediction_hybrid==0)
PAIRS['unlabeled_alert_added']=(PAIRS.label<0)&(PAIRS.prediction_neural==0)&(PAIRS.prediction_hybrid==1)
PAIRS['unlabeled_alert_removed']=(PAIRS.label<0)&(PAIRS.prediction_neural==1)&(PAIRS.prediction_hybrid==0)
transitions=['positive_gained','positive_lost','negative_alert_added','negative_alert_removed','unlabeled_alert_added','unlabeled_alert_removed']
TRANSITION_TABLE=PAIRS.groupby(['arm','seed','variant','partition'])[transitions].sum().reset_index()
TRANSITION_TABLE.to_csv(OUTPUT/'tables/routing_transition_counts_per_seed.csv',index=False)
PAIRS.to_csv(OUTPUT/'tables/paired_routing_cases_without_text.csv',index=False)
primary_pairs=PAIRS[(PAIRS.arm==ARM)&(PAIRS.seed==SEED)&(PAIRS.variant==VARIANT)].copy()
rule_rows=[]
for family,trace_key in [('crisis_affirmed','affirmed'),('crisis_negated','negated'),('safe','safe')]:
    rules=PROTOCOL['rules']['safe'] if family=='safe' else PROTOCOL['rules']['crisis']
    for rid,category,pattern in rules:
        for partition,g in primary_pairs.groupby('partition'):
            hit=g.loc[g.source_id.map(lambda x:rid in TRACES[x][trace_key]).astype(bool)]
            exclusive=hit.loc[hit.source_id.map(lambda x:len(TRACES[x][trace_key])==1).astype(bool)]
            rule_rows.append(dict(rule_id=rid,family=family,category=category,partition=partition,n_hit=len(hit),
                n_source_positive=int((hit.label==1).sum()),n_source_negative=int((hit.label==0).sum()),
                n_unknown=int((hit.label<0).sum()),n_exclusive_hit=len(exclusive),
                exclusive_negative_alerts_added=int(exclusive.negative_alert_added.sum()),
                **{c:int(hit[c].sum()) for c in transitions}))
RULE_TABLE=pd.DataFrame(rule_rows);RULE_TABLE.to_csv(OUTPUT/'tables/primary_rule_associations.csv',index=False)
print('Primary exact routing transitions:')
print(TRANSITION_TABLE[(TRANSITION_TABLE.arm==ARM)&(TRANSITION_TABLE.seed==SEED)].to_string(index=False))
print('Largest associated rule contributions on source-labeled negatives (overlapping hits):')
print(RULE_TABLE[(RULE_TABLE.family=='crisis_affirmed')&(RULE_TABLE.negative_alert_added>0)].sort_values('negative_alert_added',ascending=False).head(15).to_string(index=False))
# Optional local qualitative review file: already evaluated errors, never a new rule-tuning set.
review=primary_pairs[(primary_pairs.negative_alert_added)|(primary_pairs.positive_lost)].merge(E[['source_id','text']],on='source_id')
review.to_csv(PRIVATE/'already_inspected_rule_error_review.csv',index=False)
print('Local source-text review file (not in report ZIP):',PRIVATE/'already_inspected_rule_error_review.csv')


## 4. Freeze a matched practical benchmark for the preselected primary model
The same deterministically selected message IDs are used for FP32/INT8 and neural/hybrid. This is a timing subsample of already evaluated data, not another classifier test set. The FP32 timing uses the same threshold as the selected INT8 model to isolate computation and routing; it is not a newly selected FP32 deployment configuration.

Every configuration runs in a separate subprocess so peak RSS is not contaminated by a previously loaded model. RSS is total benchmark-process memory, including Python/tokenizer/runtime, NOT model-only memory or phone RAM. Session initialization is reported separately and may benefit from OS file caching; it is not a cold phone-app launch measurement. Warm per-message timings include actual local tokenization unless symbolic routing resolves the message first.


In [ ]:
model_specs={}
for variant in ['fp32',VARIANT]:
    directory,files=model_files(RUN,ARM,SEED,variant)
    key=f'{ARM}__{SEED}__{variant}'
    expected=EXTERNAL_PLAN['jobs'][key]['files']
    actual={str(p.relative_to(RUN)):file_sha(p) for p in files}
    if actual!=expected:raise RuntimeError(f'Frozen model/tokenizer bytes differ from the external run: {key}')
    model_specs[variant]=dict(directory=str(directory),model_path=str(directory/f'{variant}.onnx'),files=actual)
TOKENIZER=AutoTokenizer.from_pretrained(model_specs[VARIANT]['directory'],local_files_only=True,trust_remote_code=False)
benchmark_cases=[]
for profile,g in E.groupby('partition'):
    ids=sorted(g.source_id,key=lambda x:hashlib.sha256((str(ORDER_SEED)+'|'+x).encode()).hexdigest())[:MESSAGES_PER_PROFILE]
    subset=E.set_index('source_id').loc[ids]
    for row in subset.itertuples():
        enc=TOKENIZER(row.text,padding='max_length',truncation=True,max_length=MAX_LENGTH)
        i=int(E.index[E.source_id==row.Index][0]);ref={}
        for variant in ['fp32',VARIANT]:
            path=EXTERNAL/'predictions'/f'{ARM}__{SEED}__{variant}.npy'
            if not path.exists():raise FileNotFoundError(f'Frozen external probability cache missing: {path}')
            ref[variant]=dict(probability=float(np.load(path)[i]))
        benchmark_cases.append(dict(source_id=row.Index,text=row.text,text_hash=row.text_hash,partition=profile,
            label=int(row.label),group_id=row.group_id,gate_v2=int(row.gate_v2),input_ids=enc['input_ids'],attention_mask=enc['attention_mask'],reference=ref))
if not benchmark_cases:raise RuntimeError('No benchmark cases')
HOST=dict(platform=platform.platform(),processor=platform.processor(),cpu_count=os.cpu_count(),
    threads=THREADS,python=sys.version,onnxruntime=ort.__version__,provider='CPUExecutionProvider',
    transformers=importlib.metadata.version('transformers'))
BENCHMARK_PLAN=dict(version='practical_impact_v1',implementation_hash='e6bc07f7aeca78bb34c225b052ffe106a47ff850523fcdc99cec37076334c7ff',source_external_hash=EXPECTED_EXTERNAL_HASH,
    source_selection_hash=EXPECTED_SELECTION_HASH,primary=PRIMARY,model_specs={k:v['files'] for k,v in model_specs.items()},
    cases=[{k:v for k,v in c.items() if k!='text'} for c in benchmark_cases],raw_text_hashes={c['source_id']:c['text_hash'] for c in benchmark_cases},
    host=HOST,max_length=MAX_LENGTH,messages_per_profile=MESSAGES_PER_PROFILE,repeats=REPEATS,
    warmup_messages=WARMUP_MESSAGES,order_seed=ORDER_SEED,run_cpu_benchmark=RUN_CPU_BENCHMARK,
    analysis_scope='post-hoc attribution and new matched-host performance benchmark; not new model selection or clinical validation')
EVIDENCE_HASH=freeze(OUTPUT/'manifests/practical_plan_frozen.json',BENCHMARK_PLAN)
print('Practical benchmark frozen:',EVIDENCE_HASH)
print('Profiles:',pd.Series([c['partition'] for c in benchmark_cases]).value_counts().to_dict())
print('CPU tasks:',len(set(c['partition'] for c in benchmark_cases))*4,'; repeats per task:',REPEATS)


In [ ]:
RUNNER_SOURCE='"""Standalone isolated-process timing worker embedded in the notebook."""\nimport argparse,json,time,math,os,sys,resource,random,hashlib,re\nfrom pathlib import Path\nimport numpy as np\nimport onnxruntime as ort\nfrom transformers import AutoTokenizer\n\ndef build_gate(spec):\n    # Rebuild from frozen data; never execute source code read from an attachment.\n    required={\'crisis\',\'safe\',\'negators\',\'contrast\',\'risk\',\'v1_crisis\',\'v1_safe\'}\n    if not required<=set(spec): raise ValueError(\'Frozen rule specification is incomplete\')\n    C=[(i,re.compile(p)) for i,c,p in spec[\'crisis\']]\n    S=[(i,re.compile(p)) for i,c,p in spec[\'safe\']]\n    negators=set(spec[\'negators\']); contrast=re.compile(spec[\'contrast\']); risk=re.compile(spec[\'risk\'])\n    def normalize(text):\n        s=str(text).lower()\n        for ch in ["\'",\'’\',\'`\',\'‘\']: s=s.replace(ch,\'\')\n        return re.sub(r\'\\s+\',\' \',re.sub(r\'[^a-z0-9\\s]\',\' \',s)).strip()\n    def route(text,version=\'v2\',trace=False):\n        s=normalize(text)\n        if version==\'v1\':\n            for p in spec[\'v1_safe\']:\n                if normalize(p) in s: return (0,\'legacy_safe\') if trace else 0\n            for p in spec[\'v1_crisis\']:\n                if normalize(p) in s: return (1,\'legacy_crisis\') if trace else 1\n            return (-1,\'uncertain\') if trace else -1\n        aff=[]; neg=[]; safe=[]; spans=[]\n        for rid,rx in C:\n            for m in rx.finditer(s):\n                (neg if any(w in negators for w in s[:m.start()].split()[-4:]) else aff).append(rid)\n        for rid,rx in S:\n            for m in rx.finditer(s): safe.append(rid); spans.append(m.span())\n        if aff and safe: out,reason=-1,\'conflict_to_stage2\'\n        elif aff: out,reason=1,\'crisis\'\n        elif neg: out,reason=-1,\'negated_crisis_to_stage2\'\n        elif safe:\n            residual=s\n            for a,b in sorted(spans,reverse=True): residual=residual[:a]+\' \'+residual[b:]\n            if contrast.search(s) or risk.search(residual): out,reason=-1,\'safe_with_residual_risk_to_stage2\'\n            else: out,reason=0,\'safe_context\'\n        else: out,reason=-1,\'uncertain\'\n        return (out,reason) if trace else out\n    return route\n\n\ndef canonical_digest(x):return hashlib.sha256(json.dumps(x,sort_keys=True,default=str).encode()).hexdigest()\n\ndef peak_rss_mb():\n    x=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss\n    return x/(1024**2) if sys.platform==\'darwin\' else x/1024\n\ndef current_rss_mb():\n    try:\n        pages=int(Path(\'/proc/self/statm\').read_text().split()[1])\n        return pages*os.sysconf(\'SC_PAGE_SIZE\')/(1024**2)\n    except (OSError,ValueError,IndexError):return None\n\n\ndef run_task(task):\n    import re\n    gate=build_gate(task[\'rules\'])\n    cases=task[\'cases\'];variant=task[\'variant\'];mode=task[\'mode\'];maxlen=task[\'max_length\']\n    start=time.perf_counter_ns()\n    tokenizer=AutoTokenizer.from_pretrained(task[\'tokenizer_dir\'],local_files_only=True,trust_remote_code=False)\n    tokenizer_load_ms=(time.perf_counter_ns()-start)/1e6\n    before=current_rss_mb()\n    so=ort.SessionOptions();so.intra_op_num_threads=int(task[\'threads\'])\n    start=time.perf_counter_ns()\n    sess=ort.InferenceSession(task[\'model_path\'],sess_options=so,providers=[\'CPUExecutionProvider\'])\n    session_init_ms=(time.perf_counter_ns()-start)/1e6\n    after=current_rss_mb();names={x.name for x in sess.get_inputs()}\n    if names!={\'input_ids\',\'attention_mask\'}:raise ValueError(\'Benchmark worker expects frozen DistilBERT input_ids/attention_mask\')\n    # Verify frozen gate and tokenization outside timing before benchmarking.\n    for case in cases:\n        if gate(case[\'text\'])!=case[\'gate_v2\']:raise RuntimeError(\'Rule parity failed before timing\')\n        enc=tokenizer(case[\'text\'],padding=\'max_length\',truncation=True,max_length=maxlen)\n        if enc[\'input_ids\']!=case[\'input_ids\'] or enc[\'attention_mask\']!=case[\'attention_mask\']:\n            raise RuntimeError(\'Tokenizer parity failed before timing\')\n    def execute(case,measured):\n        t0=time.perf_counter_ns();gate_ns=0;token_ns=0;onnx_ns=0;p=None\n        if mode==\'hybrid\':\n            start=time.perf_counter_ns();route=gate(case[\'text\']);gate_ns=time.perf_counter_ns()-start\n        else:route=-1\n        invoked=route<0\n        if invoked:\n            start=time.perf_counter_ns()\n            enc=tokenizer(case[\'text\'],padding=\'max_length\',truncation=True,max_length=maxlen,return_tensors=\'np\')\n            feed={k:np.asarray(enc[k],dtype=np.int64) for k in names}\n            token_ns=time.perf_counter_ns()-start\n            start=time.perf_counter_ns();raw=np.asarray(sess.run([\'logits\'],feed)[0],dtype=np.float64);onnx_ns=time.perf_counter_ns()-start\n            if raw.shape!=(1,2):raise RuntimeError(\'Expected two frozen classifier logits\')\n            raw-=raw.max(axis=1,keepdims=True);exp=np.exp(raw);p=float((exp/exp.sum(axis=1,keepdims=True))[0,1])\n            decision=int(p>=task[\'tau\'])\n        else:decision=int(route)\n        total_ns=time.perf_counter_ns()-t0\n        reference=case[\'reference\'][variant]\n        expected=int(reference[\'probability\']>=task[\'tau\']) if route<0 else int(route)\n        if decision!=expected:raise RuntimeError(\'Decision parity failed; timing run cannot validate a changed classifier\')\n        if p is not None and not math.isclose(p,reference[\'probability\'],rel_tol=1e-5,abs_tol=1e-5):\n            raise RuntimeError(\'Numerical probability parity failed against external-run cache\')\n        if not measured:return None\n        return dict(source_id=case[\'source_id\'],invoked=invoked,decision=decision,\n                    total_ms=total_ns/1e6,gate_ms=gate_ns/1e6,tokenization_ms=token_ns/1e6,\n                    onnx_ms=onnx_ns/1e6 if invoked else None)\n    for i in range(task[\'warmup_messages\']):execute(cases[i%len(cases)],False)\n    records=[]\n    for repeat in range(task[\'repeats\']):\n        order=list(range(len(cases)));random.Random(task[\'order_seed\']+repeat).shuffle(order)\n        for seq,i in enumerate(order):\n            row=execute(cases[i],True);row.update(repeat=repeat,sequence=seq);records.append(row)\n    return dict(task_hash=task[\'task_hash\'],profile=task[\'profile\'],variant=variant,mode=mode,\n        threads=task[\'threads\'],repeats=task[\'repeats\'],n_unique=len(cases),warmup_messages=task[\'warmup_messages\'],\n        tokenizer_load_ms=tokenizer_load_ms,session_init_ms=session_init_ms,rss_before_session_mb=before,\n        rss_after_session_mb=after,peak_process_rss_mb=peak_rss_mb(),records=records,\n        timing_scope=\'warm per-message routing + tokenization + tensor preparation + ONNX execution + probability/threshold; excludes session/tokenizer initialization, UI and network\',\n        environment=dict(platform=sys.platform,python=sys.version,ort=ort.__version__,provider=\'CPUExecutionProvider\'))\n\n\ndef main():\n    parser=argparse.ArgumentParser();parser.add_argument(\'task\');parser.add_argument(\'output\');args=parser.parse_args()\n    task=json.loads(Path(args.task).read_text());claimed=task.pop(\'task_hash\')\n    if canonical_digest(task)!=claimed:raise RuntimeError(\'Timing task hash mismatch\')\n    task[\'task_hash\']=claimed\n    result=run_task(task);out=Path(args.output);tmp=out.with_suffix(\'.tmp\')\n    tmp.write_text(json.dumps(result,indent=1));tmp.replace(out)\n\nif __name__==\'__main__\':main()\n'
RUNNER_PATH=PRIVATE/'benchmark_worker.py'
RUNNER_PATH.write_text(RUNNER_SOURCE)


## 5. Run isolated CPU tasks and summarize paired timings
A task is cached only if its full hash matches. No parallel timing tasks are launched. All variants/modes share identical cases within each profile. Task order is fixed-randomized to reduce systematic order effects, though host scheduling and thermal variation remain possible. Unsuccessful decision/probability parity stops a task instead of treating a changed runtime as equivalent.


In [ ]:
CPU_RAW=[];CPU_META=[]
if RUN_CPU_BENCHMARK:
    tasks=[]
    for profile in sorted({c['partition'] for c in benchmark_cases}):
        cases=[c for c in benchmark_cases if c['partition']==profile]
        for variant in ['fp32',VARIANT]:
            for mode in ['neural','hybrid']:
                task=dict(profile=profile,variant=variant,mode=mode,cases=cases,rules=PROTOCOL['rules'],
                    model_path=model_specs[variant]['model_path'],tokenizer_dir=model_specs[variant]['directory'],
                    max_length=MAX_LENGTH,tau=TAU,threads=THREADS,repeats=REPEATS,warmup_messages=WARMUP_MESSAGES,
                    order_seed=ORDER_SEED,evidence_hash=EVIDENCE_HASH)
                task['task_hash']=digest(task);tasks.append(task)
    np.random.default_rng(ORDER_SEED).shuffle(tasks)
    env=os.environ.copy();env['TOKENIZERS_PARALLELISM']='false'
    for n,task in enumerate(tasks,1):
        tag=task['profile']+'__'+task['variant']+'__'+task['mode']
        result=OUTPUT/'cpu_runs'/f'{tag}.json';taskfile=PRIVATE/f'{tag}.task.json'
        atomic_json(taskfile,task)
        if result.exists():
            if json.loads(result.read_text())['task_hash']!=task['task_hash']:raise RuntimeError('CPU cache task mismatch')
            print(f'CPU {n}/{len(tasks)} cached: {tag}',flush=True)
        else:
            print(f'CPU {n}/{len(tasks)} running: {tag}',flush=True)
            run=subprocess.run([sys.executable,str(RUNNER_PATH),str(taskfile),str(result)],env=env,text=True,capture_output=True)
            if run.returncode:
                raise RuntimeError(f'CPU benchmark failed for {tag}: '+run.stderr[-5000:])
        obj=json.loads(result.read_text())
        for record in obj['records']:CPU_RAW.append(dict(profile=obj['profile'],variant=obj['variant'],mode=obj['mode'],**record))
        CPU_META.append({k:v for k,v in obj.items() if k!='records'})
    cpu=pd.DataFrame(CPU_RAW);meta=pd.DataFrame(CPU_META)
    cpu.to_csv(OUTPUT/'tables/cpu_message_timings.csv',index=False)
    meta.to_csv(OUTPUT/'tables/cpu_initialization_memory.csv',index=False)
    summary=[]
    for (profile,variant,mode),g in cpu.groupby(['profile','variant','mode']):
        invoked=g[g.invoked]
        summary.append(dict(profile=profile,variant=variant,mode=mode,n_unique=g.source_id.nunique(),
            measurements=len(g),stage2_rate=float(g.invoked.mean()),median_total_ms=float(g.total_ms.median()),
            mean_total_ms=float(g.total_ms.mean()),p95_total_ms=float(g.total_ms.quantile(.95)),
            median_invoked_onnx_ms=float(invoked.onnx_ms.median()) if len(invoked) else None,
            median_gate_ms=float(g.gate_ms.median()),median_tokenization_ms=float(g.tokenization_ms.median())))
    CPU_SUMMARY=pd.DataFrame(summary);CPU_SUMMARY.to_csv(OUTPUT/'tables/cpu_timing_summary.csv',index=False)
    paired=[]
    # Pair the same message/repeat, not unrelated traffic examples.
    for profile in cpu.profile.unique():
        for variant in ['fp32',VARIANT]:
            g=cpu[(cpu.profile==profile)&(cpu.variant==variant)]
            a=g[g['mode']=='neural'];b=g[g['mode']=='hybrid']
            pair=a[['source_id','repeat','total_ms']].merge(b[['source_id','repeat','total_ms']],on=['source_id','repeat'],suffixes=('_neural','_hybrid'),validate='one_to_one')
            paired.append(dict(profile=profile,variant=variant,comparison='hybrid_vs_neural',paired_measurements=len(pair),
                mean_paired_delta_ms=float((pair.total_ms_hybrid-pair.total_ms_neural).mean()),
                ratio_of_means=float(pair.total_ms_hybrid.mean()/pair.total_ms_neural.mean())))
        for mode in ['neural','hybrid']:
            g=cpu[(cpu.profile==profile)&(cpu['mode']==mode)]
            a=g[g.variant=='fp32'];b=g[g.variant==VARIANT]
            pair=a[['source_id','repeat','total_ms']].merge(b[['source_id','repeat','total_ms']],on=['source_id','repeat'],suffixes=('_fp32','_int8'),validate='one_to_one')
            paired.append(dict(profile=profile,variant=mode,comparison='int8_vs_fp32',paired_measurements=len(pair),
                mean_paired_delta_ms=float((pair.total_ms_int8-pair.total_ms_fp32).mean()),
                ratio_of_means=float(pair.total_ms_int8.mean()/pair.total_ms_fp32.mean())))
    pd.DataFrame(paired).to_csv(OUTPUT/'tables/cpu_paired_comparisons.csv',index=False)
    print(CPU_SUMMARY.round(4).to_string(index=False))
else:print('CPU timing skipped by predeclared configuration; attribution and phone handoff still available.')


## 6. Prepare the same benchmark for your existing Flutter/ONNX app
The adapter packet includes a frozen Dart router and a callback-based Dart timing function. Connect your existing on-device ONNX call and tokenizer to the callbacks, load each of the two specified models, and run neural/hybrid on each profile. The function verifies rule/tokenization/probability parity before accepting timing records. Without an on-device tokenizer callback, output is explicitly labeled pretokenized and cannot substantiate end-to-end claims.

Weights are excluded by default because you already have them in the attached model output. Their exact input paths, relative asset names and hashes are listed. Set `INCLUDE_PHONE_WEIGHTS=True` only to bundle the TWO primary graphs and their sidecars; this is packaging, not another model search. The separate device packet contains the small public-text sample needed for routing/tokenization parity; keep it as a private research input and follow the original source terms. It is not included in the ordinary reports ZIP.

The Dart adapter is not a standalone mobile app. Phone measurements require running your app on the actual device. No phone latency, phone RAM or battery result is generated by Kaggle.


In [ ]:
DART_GATE_SOURCE='import \'dart:convert\';\n\nclass GateTrace {\n  final int decision;\n  final String reason;\n  final List<String> affirmed, negated, safe;\n  const GateTrace(this.decision, this.reason, this.affirmed, this.negated, this.safe);\n}\n\nclass FrozenGate {\n  final Map<String, dynamic> spec;\n  late final List<List<dynamic>> crisis, safe;\n  late final Set<String> negators;\n  late final RegExp contrast, risk;\n  FrozenGate(this.spec) {\n    crisis = (spec[\'crisis\'] as List).map((x) => [x[0] as String, RegExp(x[2] as String)]).toList();\n    safe = (spec[\'safe\'] as List).map((x) => [x[0] as String, RegExp(x[2] as String)]).toList();\n    negators = (spec[\'negators\'] as List).cast<String>().toSet();\n    contrast = RegExp(spec[\'contrast\'] as String);\n    risk = RegExp(spec[\'risk\'] as String);\n  }\n  factory FrozenGate.fromJson(String s) => FrozenGate(jsonDecode(s) as Map<String, dynamic>);\n  String normalize(String text) {\n    var s = text.toLowerCase();\n    for (final ch in ["\'", \'’\', \'`\', \'‘\']) { s = s.replaceAll(ch, \'\'); }\n    return s.replaceAll(RegExp(r\'[^a-z0-9\\s]\'), \' \').replaceAll(RegExp(r\'\\s+\'), \' \').trim();\n  }\n  GateTrace trace(String text) {\n    final t = normalize(text);\n    final affirmed = <String>[], negated = <String>[], hits = <String>[];\n    final spans = <List<int>>[];\n    for (final rule in crisis) {\n      for (final m in (rule[1] as RegExp).allMatches(t)) {\n        final prior = t.substring(0, m.start).trim();\n        final words = prior.isEmpty ? <String>[] : prior.split(RegExp(r\'\\s+\'));\n        final start = words.length > 4 ? words.length - 4 : 0;\n        final isNegated = words.sublist(start).any(negators.contains);\n        (isNegated ? negated : affirmed).add(rule[0] as String);\n      }\n    }\n    for (final rule in safe) {\n      for (final m in (rule[1] as RegExp).allMatches(t)) {\n        hits.add(rule[0] as String); spans.add([m.start, m.end]);\n      }\n    }\n    int decision; String reason;\n    if (affirmed.isNotEmpty && hits.isNotEmpty) { decision = -1; reason = \'conflict_to_stage2\'; }\n    else if (affirmed.isNotEmpty) { decision = 1; reason = \'crisis\'; }\n    else if (negated.isNotEmpty) { decision = -1; reason = \'negated_crisis_to_stage2\'; }\n    else if (hits.isNotEmpty) {\n      var residual = t;\n      spans.sort((a, b) => a[0] == b[0] ? b[1].compareTo(a[1]) : b[0].compareTo(a[0]));\n      for (final span in spans) {\n        final a = span[0] > residual.length ? residual.length : span[0];\n        final b = span[1] > residual.length ? residual.length : span[1];\n        residual = residual.substring(0, a) + \' \' + residual.substring(b);\n      }\n      if (contrast.hasMatch(t) || risk.hasMatch(residual)) { decision = -1; reason = \'safe_with_residual_risk_to_stage2\'; }\n      else { decision = 0; reason = \'safe_context\'; }\n    } else { decision = -1; reason = \'uncertain\'; }\n    return GateTrace(decision, reason, affirmed.toSet().toList()..sort(), negated.toSet().toList()..sort(), hits.toSet().toList()..sort());\n  }\n  int route(String text) => trace(text).decision;\n}\n'
DART_BENCH_SOURCE="import 'dart:math' as math;\nimport 'frozen_gate.dart';\n\ntypedef RunLogits = Future<List<double>> Function(List<int> ids, List<int> mask);\ntypedef TokenizeText = Future<Map<String, List<int>>> Function(String text, int maxLength);\n\n// Supply your EXISTING on-device ONNX and tokenizer calls; this is not a new app.\n// Load one frozen model/session before calling this function. Do not time session load here.\nFuture<List<Map<String, dynamic>>> runFrozenBenchmark({\n  required FrozenGate gate,\n  required List<Map<String, dynamic>> cases,\n  required String variant,\n  required String mode,\n  required double tau,\n  required int maxLength,\n  required RunLogits runLogits,\n  required Map<String, dynamic> context,\n  TokenizeText? tokenize,\n  int repeats = 5,\n  int warmupMessages = 10,\n  int orderSeed = 20261111,\n}) async {\n  if (!['neural', 'hybrid'].contains(mode)) { throw ArgumentError('mode must be neural or hybrid'); }\n  if (cases.isEmpty) { throw ArgumentError('No cases'); }\n  for (final key in ['device_id', 'os', 'runtime', 'provider', 'threads', 'thermal_state', 'evidence_plan_hash']) {\n    if (!context.containsKey(key)) { throw ArgumentError('Missing context: $key'); }\n  }\n  bool same(List<int> a, List<int> b) {\n    if (a.length != b.length) return false;\n    for (var i = 0; i < a.length; i++) { if (a[i] != b[i]) return false; }\n    return true;\n  }\n  for (final c in cases) {\n    if (gate.route(c['text'] as String) != c['gate_v2']) { throw StateError('Frozen gate parity failed'); }\n    if (tokenize != null) {\n      final enc = await tokenize(c['text'] as String, maxLength);\n      if (!same(enc['input_ids']!, (c['input_ids'] as List).cast<int>()) ||\n          !same(enc['attention_mask']!, (c['attention_mask'] as List).cast<int>())) {\n        throw StateError('Tokenizer parity failed; do not change the tokenization protocol');\n      }\n    }\n  }\n  Future<Map<String, dynamic>> one(Map<String, dynamic> c) async {\n    final total = Stopwatch()..start();\n    var gateUs = 0, tokenUs = 0, modelUs = 0;\n    var route = -1;\n    if (mode == 'hybrid') { final sw = Stopwatch()..start(); route = gate.route(c['text'] as String); sw.stop(); gateUs = sw.elapsedMicroseconds; }\n    double? probability; int decision;\n    if (route < 0) {\n      Map<String, List<int>> enc;\n      if (tokenize != null) {\n        final sw = Stopwatch()..start(); enc = await tokenize(c['text'] as String, maxLength); sw.stop(); tokenUs = sw.elapsedMicroseconds;\n      } else { enc = {'input_ids': (c['input_ids'] as List).cast<int>(), 'attention_mask': (c['attention_mask'] as List).cast<int>()}; }\n      final sw = Stopwatch()..start(); final logits = await runLogits(enc['input_ids']!, enc['attention_mask']!); sw.stop(); modelUs = sw.elapsedMicroseconds;\n      if (logits.length != 2 || logits.any((x) => !x.isFinite)) { throw StateError('Expected two finite logits'); }\n      final maxLogit = math.max(logits[0], logits[1]);\n      final a = math.exp(logits[0] - maxLogit), b = math.exp(logits[1] - maxLogit);\n      probability = b / (a + b); decision = probability >= tau ? 1 : 0;\n    } else { decision = route; }\n    total.stop();\n    final reference = (c['reference'] as Map)[variant] as Map;\n    final expectedP = (reference['probability'] as num).toDouble();\n    final expected = route < 0 ? (expectedP >= tau ? 1 : 0) : route;\n    if (decision != expected || (probability != null && (probability - expectedP).abs() > 1e-5 + 1e-5 * expectedP.abs())) {\n      throw StateError('Frozen reference parity failed; investigate the device runtime before claiming equivalent behavior');\n    }\n    return <String, dynamic>{...context, 'profile': c['partition'], 'source_id': c['source_id'],\n      'variant': variant, 'mode': mode, 'tokenizer_mode': tokenize == null ? 'pretokenized' : 'on_device',\n      'timing_scope': tokenize == null ? 'routing_and_model_only' : 'routing_tokenization_model_pipeline',\n      'total_ms': total.elapsedMicroseconds / 1000, 'gate_ms': gateUs / 1000, 'tokenization_ms': tokenUs / 1000,\n      'onnx_ms': route < 0 ? modelUs / 1000 : null, 'invoked': route < 0, 'probability': probability, 'decision': decision, 'parity_verified': true};\n  }\n  for (var i = 0; i < warmupMessages; i++) { await one(cases[i % cases.length]); }\n  final rows = <Map<String, dynamic>>[];\n  for (var repeat = 0; repeat < repeats; repeat++) {\n    final order = List<int>.generate(cases.length, (i) => i)..shuffle(math.Random(orderSeed + repeat));\n    for (var sequence = 0; sequence < order.length; sequence++) {\n      rows.add({...await one(cases[order[sequence]]), 'repeat': repeat, 'sequence': sequence});\n    }\n  }\n  return rows;\n}\n"


In [ ]:
ADAPTER=OUTPUT/'phone_adapter'
(ADAPTER/'frozen_gate.dart').write_text(DART_GATE_SOURCE)
(ADAPTER/'device_benchmark.dart').write_text(DART_BENCH_SOURCE)
atomic_json(ADAPTER/'frozen_rules.json',PROTOCOL['rules'])
with (ADAPTER/'benchmark_cases.jsonl').open('w') as f:
    for case in benchmark_cases:f.write(json.dumps(case)+'\n')
phone_models={}
for variant,spec in model_specs.items():
    directory=Path(spec['directory']);weights=[k for k in spec['files'] if Path(k).suffix not in ['.json','.txt']]
    phone_models[variant]=dict(source_path=spec['model_path'],model_file=f'models/{variant}/{variant}.onnx',
        model_sha256=spec['files'][str(Path(spec['model_path']).relative_to(RUN))],
        tokenizer_source_dir=str(directory),all_frozen_file_hashes=spec['files'])
    if INCLUDE_PHONE_WEIGHTS:
        target_dir=ADAPTER/'models'/variant;target_dir.mkdir(parents=True,exist_ok=True)
        payload=[RUN/rel for rel in spec['files']]
        required=sum(p.stat().st_size for p in payload)
        if shutil.disk_usage(OUTPUT).free<required*2+128*1024**2:raise RuntimeError('Insufficient free space for optional two-model device package')
        for p in payload:
            destination=target_dir/p.relative_to(directory);destination.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,destination)
atomic_json(ADAPTER/'device_protocol.json',dict(evidence_plan_hash=EVIDENCE_HASH,primary=PRIMARY,
    models=phone_models,max_length=MAX_LENGTH,tau=TAU,padding='max_length',batch=1,
    profiles=sorted({c['partition'] for c in benchmark_cases}),modes=['neural','hybrid'],
    repeats=REPEATS,warmup_messages=WARMUP_MESSAGES,order_seed=ORDER_SEED,
    required_context=['device_id','os','runtime','provider','threads','thermal_state','evidence_plan_hash'],
    classification_scope='parity and timing on a fixed subset of previously inspected external rows; not new independent classification evaluation',
    pretokenized_scope='routing_and_model_only',on_device_tokenizer_scope='routing_tokenization_model_pipeline'))
DEVICE_COLUMNS=['device_id','os','runtime','provider','threads','thermal_state','evidence_plan_hash','profile','source_id',
    'variant','mode','tokenizer_mode','timing_scope','repeat','sequence','total_ms','gate_ms','tokenization_ms','onnx_ms',
    'invoked','probability','decision','parity_verified']
pd.DataFrame(columns=DEVICE_COLUMNS).to_csv(ADAPTER/'device_results_template.csv',index=False)
(ADAPTER/'README_phone.txt').write_text(
    'Use your existing Flutter/ONNX app. This is a tested Dart gate plus a callback-based timing adapter, not an app.\n'
    'Read device_protocol.json, frozen_rules.json and benchmark_cases.jsonl. Copy the exact two ONNX models/tokenizers from the listed source paths if weights are not bundled.\n'
    'Load one model/session per configuration. Run both modes on the exact cases for each profile. Use warmups/repeats from the protocol.\n'
    'Connect runLogits to your existing local ONNX session, returning two logits; connect tokenize to the real local tokenizer for full pipeline timing.\n'
    'Do not present pretokenized timing as end-to-end. Session load, UI rendering, network, battery and power are not measured by the adapter.\n'
    'Record actual device/OS/runtime/provider/thread/thermal context. Keep UI work and network calls outside the measured callbacks.\n'
    'Export each returned map as a CSV row using device_results_template.csv columns; null ONNX/probability values are valid for symbolically resolved messages.\n'
    'Do not record probability mismatches as successful timings: the adapter throws and the runtime needs investigation.\n'
    'Per-profile/classification counts here are from a timing subsample and do not replace the full external evaluation.\n'
    'To analyze real phone logs, attach their CSV to this notebook and set DEVICE_RESULTS_CSV before rerunning. This does not retrain or tune.\n'
    'The packet contains public source sample texts for private research measurement. Source references: https://github.com/laxmimerit/twitter-suicidal-intention-dataset and https://github.com/thu-coai/Emotional-Support-Conversation . Preserve original terms and attribution.\n')
print('Phone adapter prepared. Actual phone execution is still required.')


## 7. Optional import of actual phone logs
Nothing in this cell fabricates phone results. With no CSV supplied, it reports pending. Imported logs must match the frozen plan, cases, model variants, thresholds, labels of timing scope, expected decisions, repeats, and parity flags. Neural/hybrid comparisons are paired by the same source ID and repeat. Different devices/providers or pretokenized/on-device-tokenizer runs are never merged into one comparison.


In [ ]:
if DEVICE_RESULTS_CSV:
    device=pd.read_csv(DEVICE_RESULTS_CSV)
    required=set(DEVICE_COLUMNS)
    if not required<=set(device):raise ValueError('Device CSV missing columns: '+str(sorted(required-set(device))))
    if len(device)==0:raise ValueError('Device CSV is empty')
    if device[['device_id','os','runtime','provider','threads','thermal_state','evidence_plan_hash']].isna().any().any():raise ValueError('Missing device context values')
    if set(device.evidence_plan_hash)!={EVIDENCE_HASH}:raise ValueError('Device logs belong to a different frozen benchmark')
    if not set(device.variant)<=set(model_specs):raise ValueError('Unexpected device model variant')
    if not set(device['mode'])<={'neural','hybrid'}:raise ValueError('Unexpected device mode')
    for col in ['invoked','parity_verified']:
        values=device[col].astype(str).str.lower().map({'true':True,'false':False,'1':True,'0':False})
        if values.isna().any():raise ValueError('Invalid device boolean column: '+col)
        device[col]=values
    if not device.parity_verified.all():raise ValueError('Some device rows did not pass parity')
    case_map={c['source_id']:c for c in benchmark_cases}
    for row in device.itertuples():
        if row.source_id not in case_map:raise ValueError('Unknown device case ID')
        case=case_map[row.source_id]
        if row.profile!=case['partition']:raise ValueError('Device case/profile mismatch')
        if row.tokenizer_mode=='on_device':
            if row.timing_scope!='routing_tokenization_model_pipeline':raise ValueError('Wrong timing-scope label')
        elif row.tokenizer_mode=='pretokenized':
            if row.timing_scope!='routing_and_model_only':raise ValueError('Pretokenized result mislabeled as end-to-end')
        else:raise ValueError('Unknown tokenizer mode')
        route=case['gate_v2'] if row.mode=='hybrid' else -1
        if row.invoked!=(route<0):raise ValueError('Stage-2 invocation does not match frozen routing')
        p=case['reference'][row.variant]['probability'];expected=int(p>=TAU) if route<0 else route
        if int(row.decision)!=expected:raise ValueError('Device decision differs from frozen reference')
        if route<0 and (not np.isfinite(row.probability) or not np.isclose(row.probability,p,atol=1e-5,rtol=1e-5)):
            raise ValueError('Device probability differs from frozen reference')
        if any(not np.isfinite(v) or v<0 for v in [row.total_ms,row.gate_ms,row.tokenization_ms]):raise ValueError('Invalid or negative device timing')
        if route<0 and (not np.isfinite(row.onnx_ms) or row.onnx_ms<0):raise ValueError('Invalid invoked-only model timing')
    contexts=['device_id','os','runtime','provider','threads','thermal_state','tokenizer_mode','timing_scope']
    groups=contexts+['profile','variant','mode']
    ds=[]
    for key,g in device.groupby(groups,dropna=False):
        if g.duplicated(['source_id','repeat']).any():raise ValueError('Duplicate device message/repeat')
        profile=key[-3];expected_ids={c['source_id'] for c in benchmark_cases if c['partition']==profile}
        if set(g['repeat'])!=set(range(REPEATS)):raise ValueError('Incomplete device repeats')
        for repeat,block in g.groupby('repeat'):
            if set(block.source_id)!=expected_ids:raise ValueError('Incomplete device case set')
        ds.append(dict(zip(groups,key),median_total_ms=float(g.total_ms.median()),mean_total_ms=float(g.total_ms.mean()),
            p95_total_ms=float(g.total_ms.quantile(.95)),stage2_rate=float(g.invoked.mean()),measurements=len(g)))
    pd.DataFrame(ds).to_csv(OUTPUT/'tables/actual_phone_timing_summary.csv',index=False)
    paired=[]
    for key,g in device.groupby(contexts+['profile','variant'],dropna=False):
        a=g[g['mode']=='neural'];b=g[g['mode']=='hybrid']
        if not len(a) or not len(b):continue
        pair=a[['source_id','repeat','total_ms']].merge(b[['source_id','repeat','total_ms']],on=['source_id','repeat'],suffixes=('_neural','_hybrid'),validate='one_to_one')
        paired.append(dict(zip(contexts+['profile','variant'],key),paired_measurements=len(pair),
            mean_delta_ms=float((pair.total_ms_hybrid-pair.total_ms_neural).mean()),ratio_of_means=float(pair.total_ms_hybrid.mean()/pair.total_ms_neural.mean())))
    pd.DataFrame(paired).to_csv(OUTPUT/'tables/actual_phone_hybrid_vs_neural.csv',index=False)
    print('Imported verified actual phone logs. Missing variants/modes do not establish a full four-way comparison.')
else:print('Actual phone measurements: PENDING. Kaggle CPU results are not phone measurements.')


## 8. Save a concise evidence report and two separate packages
The ordinary report package is small and contains no weights or source texts. The optional device package contains the small public input sample and Dart adapters, plus weights only if explicitly enabled. No result is automatically described as a speedup: a ratio above 1 means slower on this host/traffic profile.


In [ ]:
lines=[
 'CrisisQ practical impact: frozen models, no retraining, no threshold/rule changes.',
 'Attribution is post-hoc analysis of already inspected external data. It is not an additional untouched classification test.',
 'Per-rule hit effects overlap; only exact pipeline transitions are disjoint paired outcomes.',
 'Kaggle timings measure this CPU host, not a phone. RSS is full isolated-process memory, not model-only or mobile RAM.',
 'End-to-end inference-pipeline timings exclude initialization, UI, network and battery effects; those require separate measurements.',
 'No superiority or clinical benefit is assumed. Interpret recovered positives alongside added negative alerts.',
 'Historical source-label/overlap limitations from the external evaluation remain unchanged.'
]
for row in TRANSITION_TABLE[(TRANSITION_TABLE.arm==ARM)&(TRANSITION_TABLE.seed==SEED)].itertuples():
    lines.append(f'Primary {row.partition}: positive gains={row.positive_gained}, positive losses={row.positive_lost}, added source-negative alerts={row.negative_alert_added}, removed source-negative alerts={row.negative_alert_removed}.')
if RUN_CPU_BENCHMARK:
    for row in CPU_SUMMARY.itertuples():
        lines.append(f'HOST CPU {row.profile}/{row.variant}/{row.mode}: mean={row.mean_total_ms:.4f} ms, median={row.median_total_ms:.4f} ms, Stage-2 rate={row.stage2_rate:.4f}.')
lines.append('Phone execution status: '+('actual logs imported under strict contract' if DEVICE_RESULTS_CSV else 'pending; adapter packet exported only'))
(OUTPUT/'tables/practical_claim_check.txt').write_text('\n'.join(lines))
# Recheck both selected model inventories after timing; no artifacts may change.
for variant,spec in model_specs.items():
    for rel,sha in spec['files'].items():
        if file_sha(RUN/rel)!=sha:raise RuntimeError('Frozen input artifact changed during benchmark')
atomic_json(OUTPUT/'manifests/practical_complete.json',dict(evidence_hash=EVIDENCE_HASH,
    source_external_hash=EXPECTED_EXTERNAL_HASH,new_models_trained=0,thresholds_changed=0,rules_changed=0,
    cpu_measured=RUN_CPU_BENCHMARK,actual_phone_logs_imported=bool(DEVICE_RESULTS_CSV)))
def zip_verified(target,files,base):
    total=sum(p.stat().st_size for p in files)
    if shutil.disk_usage(OUTPUT).free<total+128*1024**2:raise RuntimeError('Insufficient disk space for archive; raw results remain saved')
    tmp=target.with_suffix('.partial.zip')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED,compresslevel=3) as z:
        for p in files:z.write(p,p.relative_to(base))
    with zipfile.ZipFile(tmp) as z:
        if z.testzip() is not None:raise RuntimeError('Archive integrity check failed')
    tmp.replace(target);return target
report_files=[]
for folder in ['tables','manifests','cpu_runs']:
    report_files += [p for p in (OUTPUT/folder).rglob('*') if p.is_file() and not p.is_symlink()]
report=zip_verified(OUTPUT.parent/'crisisq_practical_impact_v1_reports.zip',report_files,OUTPUT)
phone=zip_verified(OUTPUT.parent/'crisisq_device_handoff_v1.zip',[p for p in ADAPTER.rglob('*') if p.is_file() and not p.is_symlink()],ADAPTER)
print('REPORTS:',report)
print('PRIVATE DEVICE HANDOFF:',phone)
print('\n'.join(lines))


**How to use these results in the revision:** report model-file size separately from measured process memory; report matched CPU timing separately from actual phone timing; show positive recoveries and negative-alert additions together; explain the rule categories associated with failures. If the router is slower or increases unwanted alerts on a profile, report that outcome. No more seed, quantizer, threshold or rule searches are part of this notebook.

Timing implementation references: [ONNX Runtime thread management](https://onnxruntime.ai/docs/performance/tune-performance/threading.html), [Python process resource measurements](https://docs.python.org/3/library/resource.html).
